# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mass Spectrometry dataset using the [`mlcroissant`](https://mlcommons.org/croissant) library, referencing all data entities by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata (as an object, not a dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List available record sets and their fields (all referenced by `@id`).

In [ ]:
# List all record sets and their fields, referencing them by @id
record_sets = dataset.metadata.record_sets
print("Available RecordSets (by @id):\n")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    if hasattr(rs, 'fields'):
        print("  Fields (@id, name, type):")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {field.name} | type: {field.data_type}")
    print()
# For convenience, create a list of record set @ids
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Extract records from each record set into pandas DataFrames. Use only `@id` values for referencing.

In [ ]:
# Loop over each record set and load its records by @id
dataframes = {}
print(f"Loading data from these RecordSets (by @id): {record_set_ids}\n")
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"RecordSet @id: {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    print(f"Rows: {len(dataframes[rs_id])}\n")

# Pick the first record set for demo
if record_set_ids:
    demo_rs_id = record_set_ids[0]
    print(f"Sample records from RecordSet @id: {demo_rs_id}")
    display(dataframes[demo_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Process the records: filter by a numeric field, normalize, group. Use only `@id` for field references. **Choose fields based on printed overview.**

In [ ]:
# Choose a numeric field and a group field by @id (edit these based on your output above)

# Example: @id fields from a typical proteomics dataset
# Replace these IDs with field @ids you printed earlier as appropriate for your dataset
numeric_field_id = None
group_field_id = None

# Inspect available columns for the first record set to help choose these fields
first_rs = dataframes[demo_rs_id]
print(f"Available columns in {demo_rs_id} (likely field @ids): {first_rs.columns.tolist()}")

# Try to auto-select a numeric field (float/int)
for col in first_rs.columns:
    if pd.api.types.is_numeric_dtype(first_rs[col]):
        numeric_field_id = col
        break

# Try to auto-select a likely group field (object, categorical)
for col in first_rs.columns:
    if col != numeric_field_id and pd.api.types.is_string_dtype(first_rs[col]):
        group_field_id = col
        break

if numeric_field_id:
    threshold = first_rs[numeric_field_id].quantile(0.75) if not first_rs[numeric_field_id].isnull().all() else 0
    filtered_df = first_rs[first_rs[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalize numeric field
    col_normed = numeric_field_id + '_normalized'
    filtered_df[col_normed] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_normed]].head())
    
    # Group by a group field, if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please review the fields and choose an appropriate one.")

## 5. Visualization
Visualize the numeric distribution and (optionally) the grouping if applicable. All axes are labeled by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(first_rs[numeric_field_id].dropna(), bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.show()
    
    # If grouped_df exists
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (by @id)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
This notebook demonstrated loading, inspecting, and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` Python library. Entity references were consistently made using their Croissant `@id` for full transparency and reproducibility. Replace selected fields with appropriate `@id` values from your overview to tailor further exploration to your research needs.